# Learning e^t via Taylor Design Matrix
Fix a = 1, discover e^t = Σ t^n / n! using least squares on varying step sizes.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math

# Fixed parameters
a     = 1          # fixed: learning e^t
t_max = 1
N_max = 6          # highest Taylor degree

# Step sizes to compare
step_sizes = [5, 10, 25, 50, 100, 500]

## Cell 2 — Helper: build design matrix and solve for c

In [ ]:
def make_design_matrix(t_arr, N):
    """Columns: t^n / n! for n = 0 ... N"""
    return np.column_stack([(t_arr ** n) / math.factorial(n) for n in range(N + 1)])

def solve(num_steps, N=N_max):
    t_arr = np.linspace(0, t_max, num_steps)
    b     = np.exp(a * t_arr)           # true values: e^t
    D     = make_design_matrix(t_arr, N)
    c     = np.linalg.lstsq(D, b, rcond=None)[0]
    cond  = np.linalg.cond(D)
    resid = np.linalg.norm(D @ c - b) / np.linalg.norm(b)
    return c, cond, resid, t_arr, D, b

## Cell 3 — Vary step sizes: coefficient recovery

In [ ]:
print(f"{'steps':<8} {'cond(D)':<14} {'rel resid':<14}", end="")
for n in range(N_max + 1):
    print(f"  c[{n}]", end="")
print()
print("-" * (8 + 14 + 14 + (N_max + 1) * 7))

results = {}
for steps in step_sizes:
    c, cond, resid, t_arr, D, b = solve(steps)
    results[steps] = (c, cond, resid)
    print(f"{steps:<8} {cond:<14.2e} {resid:<14.2e}", end="")
    for cn in c:
        print(f"  {cn:>5.3f}", end="")
    print()

print("\nTarget coefficients (all 1.0):")
print(" " * 36, end="")
for _ in range(N_max + 1):
    print(f"  {'1.000':>5}", end="")
print()

## Cell 4 — Plot: coefficient error vs step size

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: condition number vs steps
conds = [results[s][1] for s in step_sizes]
axes[0].loglog(step_sizes, conds, 'bo-')
axes[0].set_xlabel('Number of steps')
axes[0].set_ylabel('cond(D)')
axes[0].set_title('Condition number vs step size')
axes[0].grid(True, which='both', alpha=0.3)

# Right: max coefficient error vs steps
max_errs = [np.max(np.abs(results[s][0] - 1.0)) for s in step_sizes]
axes[1].loglog(step_sizes, max_errs, 'rs-')
axes[1].set_xlabel('Number of steps')
axes[1].set_ylabel('max |c[n] - 1|')
axes[1].set_title('Coefficient recovery error vs step size')
axes[1].grid(True, which='both', alpha=0.3)

plt.tight_layout()
plt.show()

## Cell 5 — Plot: predicted vs true e^t for each step size

In [ ]:
t_fine = np.linspace(0, t_max, 500)
D_fine = make_design_matrix(t_fine, N_max)
y_true = np.exp(t_fine)

plt.figure(figsize=(9, 5))
plt.plot(t_fine, y_true, 'k-', lw=2.5, label='True e^t', zorder=10)

colors = plt.cm.viridis(np.linspace(0.1, 0.9, len(step_sizes)))
for col, steps in zip(colors, step_sizes):
    c = results[steps][0]
    y_pred = D_fine @ c
    plt.plot(t_fine, y_pred, '--', color=col, lw=1.2, label=f'{steps} steps')

plt.xlabel('t')
plt.ylabel('e^t')
plt.title('e^t approximation: varying number of steps')
plt.legend(fontsize=9)
plt.tight_layout()
plt.show()

## Cell 6 — ISTA on e^t (50 steps, as baseline)

In [ ]:
def soft_threshold(z, eta):
    return np.sign(z) * np.maximum(np.abs(z) - eta, 0)

def run_ista(D, b, lam_frac=0.02, maxit=5000):
    col_norms = np.linalg.norm(D, axis=0)
    D_n = D / col_norms
    L   = np.linalg.norm(D_n, 2) ** 2
    t_s = 1.0 / L
    lam = lam_frac * np.max(np.abs(D_n.T @ b))
    x   = np.zeros(D_n.shape[1])
    for _ in range(maxit):
        grad = D_n.T @ (D_n @ x - b)
        x    = soft_threshold(x - t_s * grad, t_s * lam)
    return x / col_norms

_, _, _, t_arr, D, b = solve(50)
c_ista = run_ista(D, b)

print(f"{'n':<5} {'ISTA c[n]':>12} {'target':>10} {'error':>10}")
print("-" * 40)
for n, cn in enumerate(c_ista):
    print(f"{n:<5} {cn:>12.6f} {'1.000000':>10} {abs(cn - 1.0):>10.2e}")